# The scenarios construction via [premise](https://github.com/romainsacchi/premise)

Inital Author about premise: [romainsacchi](https://github.com/romainsacchi)

This notebook shows examples on how to use `premise` to adapt the life cycle inventory database [ecoinvent](https://www.ecoinvent.org/) for prospective environmental impact assessment.


This library extract useful information from IAM model output files (such as those of REMIND or IMAGE) and aligns inventories in the ecoinvent database accordingly.


# Use case with [brightway2](https://brightway.dev/)

`brightway2` is an open source LCA framework for Python.
To use `premise` from `brightway2`, it requires that you have an activated `brightway2` project with a `biosphere3` database as well as an ecoinvent v.3 cut-off or consequential database registered in that project.

In [ ]:
from premise import *
import bw2data
import random
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import brightway2 as bw
from brightway2 import *
from bw2data.parameters import ActivityParameter, DatabaseParameter, ProjectParameter, Group

### List of available scenarios

Some scenarios come installed with the library.
They are stored in `data/iam_ouput_files` from the root directory.
They are all within the same Shared Socio-Economic Pathway (SSP): SSP2 (nicknamed "middle of the road"), which describes a future world (in terms of GDP and demographics development, education, intergovernmental collaboration) very much in line with what has been observed historically..

But they are proposed in combination with different climate mitigation targets, called Representative Concentration Pathways (RCP).
Read more about SSPs and RCPs, [here](https://www.carbonbrief.org/explainer-how-shared-socioeconomic-pathways-explore-future-climate-change).



### Database creation from default scenarios

To create a scenario using REMIND's SSP2 Base pathway, from ecoinvent 3.5 for the year 2028, one would execute the following cell. This leads to the extraction of the database, some cleanup as well as importing a few additional inventories.

In [ ]:
bw2data.projects.set_current("PFAS-LCA")
bw2data.databases

The first time you create a premise database, *premise* will store a copy of the ecoinvent database and external inventories, to be able to skip that time-consuming step next time. If you wish to clear this cache (which is only encourage if updating premise or if encountering issues with inventories), do:

In [ ]:
clear_cache()

In [ ]:
ndb = NewDatabase(
    scenarios=[
        {"model":"image", "pathway":"SSP2-RCP26", "year":2025},
        {"model":"image", "pathway":"SSP2-RCP26", "year":2030},
        {"model":"image", "pathway":"SSP2-RCP26", "year":2035},
        {"model":"image", "pathway":"SSP2-Base", "year":2025},
        {"model":"image", "pathway":"SSP2-Base", "year":2030},
        {"model":"image", "pathway":"SSP2-Base", "year":2035},
    ],
    source_db="ecoinvent 3.8_cutoff_ecoSpold02", # <-- name of the database in the BW2 project. Must be a string.
    source_version="3.8", # <-- version of ecoinvent. Can be "3.5", "3.6", "3.7" or "3.8". Must be a string.
    key='tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo=' # <-- decryption key
    # to be requested from the library maintainers if you want ot use default scenarios included in `premise`
)

If you do not want to integrate the IAM projections in the database, but only wish to have the additional inventories, you can stop here and export the database back to Brightway or other destinations, by using the `write_db_to` methods, like so:

In [ ]:
ndb.write_db_to_brightway()

Howver, if you wish first to proceed with the IAM integration, you need to use the `update_` methods, like so for the electricity sector:

In [ ]:
ndb.update()

In [ ]:
ndb = NewDatabase(
    scenarios=[
        #{"model":"remind", "pathway":"SSP1-NDC", "year":2030},
        #{"model":"remind", "pathway":"SSP1-NDC", "year":2035},
        {"model":"TIAM-UCL", "pathway":"SSP2-RCP45", "year":2030},
        {"model":"TIAM-UCL", "pathway":"SSP2-RCP45", "year":2035},
    ],
    source_db="ecoinvent 3.8_cutoff_ecoSpold02", # <-- name of the database in the BW2 project. Must be a string.
    source_version="3.8", # <-- version of ecoinvent. Can be "3.5", "3.6", "3.7" or "3.8". Must be a string.
    key='tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo=' # <-- decryption key
    # to be requested from the library maintainers if you want ot use default scenarios included in `premise`
)

In [ ]:
ndb.update()

In [ ]:
ndb.write_db_to_brightway()

In [ ]:
import sys
print(sys.version.split()[0])

In [ ]:
ndb.update_fuels()

In [ ]:
ndb.write_db_to_brightway()

or here with ecoinvent 3.XX

In [ ]:
ndb = NewDatabase(
        scenarios=[
                {"model":"remind", "pathway":"SSP2-Base", "year":2028}
            ],
        source_db="ecoinvent 3.XX cutoff", # <-- this is NEW.
        source_version="3.XX", # <-- this is NEW
        key='xxxxxxxxxxxxxxxxxxxxxxxxx'
    )

If you want to create multiple databases at once, just populate the `scenarios` list.
You will notice the key `exclude` for which we can list the transformations we do not wish to perform.
In this case, we do not wish to update the electricity sector.

In [ ]:
ndb = NewDatabase(
            scenarios=[
                {"model":"remind", "pathway":"SSP2-Base", "year":2020, "exclude": ["update_electricity"]},
                {"model":"remind", "pathway":"SSP2-Base", "year":2030},
                {"model":"remind", "pathway":"SSP2-Base", "year":2040},
                {"model":"remind", "pathway":"SSP2-Base", "year":2050},
            ],
            source_db="ecoinvent 3.XX cutoff", # <-- name of the database. Must be a string.
            source_version="3.XX", # <-- version of ecoinvent. Can be "3.5", "3.6", "3.7" or "3.7.1"
            key='xxxxxxxxxxxxxxxxxxxxxxxxx'
)

In [ ]:
ndb.update_all()

When the database is loaded and the additional inventories imported, you can apply a transformation function.
For example here, we adjust the efficiency of the power plants to the two scenarios we have loaded.
We go more in details later.

In [ ]:
ndb.update_electricity()

Or you can proceed instead to doing all the transformations available (minus any transformation you have listed in `exclude`), like so:

In [ ]:
ndb.update_all()

And then, we register these two databases back into brightway2.

In [ ]:
ndb.write_db_to_brightway()

### As a data package
Export a data package, which can be shared. Data packages cna be read by [unfold](https://github.com/polca/unfold) and databases can be reproduced on other computers.

In [ ]:
ndb.write_datapackage()

# Reports

## Scenario report

You can generate a spreadsheet report showing the main variables of the scenario you have selected to create your databases.
The report is saved in your working directory.

In [ ]:
ndb.generate_scenario_report()

## Changes report

You can generate a spreadsheet report of the changes made to the original database.
It gives an overview on:

* the datasets created
* the datasets modified
* some performance indicators
* scaling factors used to scale certain exchanges

The report is saved in your working directory.

In [ ]:
ndb.generate_change_report()